# Notebook 02 — Preprocessing: Face Crops + Audio Extraction

**Project:** AV-XAI (Explainable Multimodal Deepfake Detection)
**Input:** 30K LAV-DF clips (manifest from notebook 01)
**Output:** One `.pt` file per clip containing pre-extracted face crops + audio waveform

## What this notebook produces

For each clip, save a single `.pt` dict at `data/preprocessed/{split}/{stem}.pt`:

```python
{
    'faces':       torch.uint8  [16, 3, 224, 224],   # MTCNN-cropped face per frame
    'audio':       torch.float32 [160000],            # 10s mono @ 16kHz, padded/center-cropped
    'label_4class': int (0–3),
    'label_binary': int (0/1),
    'modify_type':  str,
    'split':        str,
    'file':         str,            # original LAV-DF path, for traceback
    'fake_periods': list,           # [[start_s, end_s], ...] for future temporal localization
    'duration':     float,          # original clip length, seconds
    'face_stats':   {detected, fallback_center_crop, total_frames}
}
```

## Why these specific shapes

- **16 frames** uniformly sampled across clip duration → matches ViViT-B/16-x2 tubelet processing
- **224×224** crops → ViViT-B/16 native input resolution (Kinetics-400 pretraining)
- **uint8** storage → 4× smaller than float32; normalize at training time with Kinetics mean/std
- **10s audio @ 16kHz** → 160,000 samples → WavLM-Large native sample rate; longer than median LAV-DF clip so most clips need padding (not truncation)

## Smoke-test pattern (intentional)

Same philosophy as notebook 01: catch problems on 1 clip before burning hours on 30K.

1. **Phase A** — Inspect 4 raw clips (resolution, fps, save thumbnails) to decide if MTCNN is needed at all
2. **Phase B** — Initialize MTCNN on GPU, smoke test on one clip
3. **Phase C** — Define `extract_faces()` and `extract_audio()` with fallbacks
4. **Phase D** — Process 15 clips (5 per split), assert tensor shapes
5. **Phase E** — Full 30K run, resumable, failure-logging
6. **Phase F** — Verification: random spot-checks, disk usage

## Disk budget

~2.4 MB faces + 640 KB audio per clip × 30K clips ≈ **72 GB total**. Fits in your 315 GB free.

## Runtime estimate (RTX A6000)

- MTCNN GPU inference: ~10 clips/sec
- Audio: trivial CPU cost (librosa is fast)
- Full 30K run: ~50–60 minutes

In [ ]:
# --- Imports and paths ---
import os, sys, json, time, traceback, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Video / audio I/O
import av                         # PyAV — already in df env
import librosa                    # audio loading + resampling
from PIL import Image

# Face detection
from facenet_pytorch import MTCNN

# Matplotlib only for the smoke-test thumbnails
import matplotlib.pyplot as plt

# --- Reproducibility ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Paths (match notebook 01) ---
PROJECT_ROOT = Path(r"C:\Users\Komal-Sch\Desktop\AV-XAI")
DATA_ROOT    = PROJECT_ROOT / "data"
LAVDF_ROOT   = DATA_ROOT / "LAV-DF"             # raw clips (matches the extracted tar)
SPLITS_DIR   = DATA_ROOT / "splits"
PREPROC_ROOT = DATA_ROOT / "preprocessed"
THUMB_DIR    = DATA_ROOT / "_thumbnails"        # for Phase A inspection

for s in ("train", "val", "test"):
    (PREPROC_ROOT / s).mkdir(parents=True, exist_ok=True)
THUMB_DIR.mkdir(parents=True, exist_ok=True)

print("LAVDF_ROOT  :", LAVDF_ROOT)
print("PREPROC_ROOT:", PREPROC_ROOT)
print("Splits dir  :", SPLITS_DIR)

# --- Preprocessing knobs ---
N_FRAMES        = 16          # frames per clip
IMG_SIZE        = 224         # ViViT-B/16 native input
MTCNN_MARGIN    = 20          # extra context pixels around detected face
AUDIO_SR        = 16_000      # WavLM-Large native sample rate
AUDIO_LEN_SEC   = 10
AUDIO_SAMPLES   = AUDIO_SR * AUDIO_LEN_SEC   # 160,000

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Phase A — Inspect 4 raw clips (do we even need MTCNN?)

LAV-DF is built on VoxCeleb2 talking-head videos. If the clips are already tightly face-cropped, MTCNN is redundant and we save ~45 minutes of GPU time by skipping it.

This cell loads one random clip per class, prints its resolution/fps/duration, and saves the first frame as a PNG thumbnail to `data/_thumbnails/`. Open those PNGs in Explorer and eyeball:

- **If the face fills most of the frame** → skip MTCNN, just resize to 224×224
- **If you see a wide shot with the face in part of the frame** → MTCNN needed (the default)

In [ ]:
# --- Load manifest and pick one clip per class ---
manifest = pd.read_csv(DATA_ROOT / "manifest.csv")
print(f"Manifest: {len(manifest):,} clips")

rng = random.Random(SEED)
samples = {}
for mt in ("real", "visual_modified", "audio_modified", "both_modified"):
    pool = manifest[manifest["modify_type"] == mt]
    samples[mt] = pool.sample(1, random_state=SEED).iloc[0]

# --- Inspect each one ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (mt, row) in zip(axes, samples.items()):
    fp = LAVDF_ROOT / row["file"]
    container = av.open(str(fp))
    vs = container.streams.video[0]
    has_audio = len(container.streams.audio) > 0

    w, h, fps = vs.width, vs.height, float(vs.average_rate)
    duration = float(container.duration / 1_000_000) if container.duration else None

    # Grab first frame
    first_frame = None
    for frame in container.decode(video=0):
        first_frame = frame.to_ndarray(format="rgb24")
        break
    container.close()

    # Save thumbnail
    thumb_fp = THUMB_DIR / f"{mt}_{Path(row['file']).stem}.png"
    Image.fromarray(first_frame).save(thumb_fp)

    ax.imshow(first_frame)
    ax.set_title(f"{mt}\n{w}×{h} @ {fps:.1f}fps\n{duration:.1f}s | audio={has_audio}", fontsize=9)
    ax.axis("off")
    print(f"{mt:<16}  {row['file']:<20}  {w}×{h}  {fps:.1f}fps  {duration:.2f}s  audio={has_audio}")

plt.tight_layout()
plt.show()
print(f"\nThumbnails saved to: {THUMB_DIR}")
print("Open them in File Explorer and decide: do clips need MTCNN face detection?")

## Phase B — Initialize MTCNN on GPU + smoke test

Defaults assume MTCNN is needed (the safe assumption). If Phase A showed clips are already face-cropped, you can set `USE_MTCNN = False` in the next cell and the pipeline falls back to a simple resize.

MTCNN config:
- `image_size=224` → output crop matches ViViT input
- `margin=20` → pixels of context around the detected face box
- `keep_all=False` → we want the single most prominent face (talking head, one speaker)
- `post_process=False` → skip MTCNN's whitening; we'll normalize at training time
- `selection_method="probability"` → pick the most confident detection

In [ ]:
# --- Toggle ---
USE_MTCNN = True   # set False if Phase A showed clips are already tightly face-cropped

# --- Init MTCNN ---
mtcnn = MTCNN(
    image_size=IMG_SIZE,
    margin=MTCNN_MARGIN,
    keep_all=False,
    post_process=False,
    select_largest=False,
    selection_method="probability",
    device=device,
)
mtcnn.eval()
print(f"MTCNN initialized on {device}")

# --- Smoke test on one clip ---
test_clip = samples["real"]["file"]
test_fp = LAVDF_ROOT / test_clip
print(f"Smoke test on: {test_fp.name}")

container = av.open(str(test_fp))
frames = []
for frame in container.decode(video=0):
    frames.append(frame.to_ndarray(format="rgb24"))
container.close()
print(f"  Decoded {len(frames)} frames")

# Pick the middle frame and run MTCNN
mid = frames[len(frames)//2]
pil = Image.fromarray(mid)
with torch.no_grad():
    face = mtcnn(pil)        # returns [3, 224, 224] float tensor or None

if face is None:
    print("  !! MTCNN found no face on the middle frame. Investigate before continuing.")
else:
    print(f"  Face tensor shape: {tuple(face.shape)}, dtype: {face.dtype}, range: [{face.min():.1f}, {face.max():.1f}]")
    # Show the crop
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(mid);                    axes[0].set_title("Original middle frame"); axes[0].axis("off")
    axes[1].imshow(face.byte().permute(1,2,0).cpu()); axes[1].set_title("MTCNN crop (224×224)"); axes[1].axis("off")
    plt.tight_layout(); plt.show()

## Phase C — Define `extract_faces()` and `extract_audio()`

**`extract_faces(video_path)` →** `(faces [16,3,224,224] uint8, stats dict)`
- Decodes all video frames via PyAV
- Uniformly samples 16 indices across the full clip
- Runs MTCNN per sampled frame (or simple resize if `USE_MTCNN=False`)
- **Fallback:** if MTCNN finds no face on a frame, use a center crop of the frame resized to 224×224. Logged in `stats`.

**`extract_audio(video_path)` →** `audio [160000] float32`
- Loads audio via librosa at 16 kHz mono
- Pads with zeros if shorter than 10s
- Center-crops to 10s if longer
- Returns silent array (zero) if the clip has no audio stream (shouldn't happen with LAV-DF; logged just in case)

In [ ]:
# --- Helpers ---

def _uniform_indices(n_total: int, n_pick: int) -> list:
    """Return n_pick evenly-spaced indices in [0, n_total). Robust to n_total < n_pick."""
    if n_total <= 0:
        return [0] * n_pick
    if n_total <= n_pick:
        # repeat last frame to fill
        idx = list(range(n_total)) + [n_total - 1] * (n_pick - n_total)
        return idx
    step = n_total / n_pick
    return [int(i * step) for i in range(n_pick)]


def _center_crop_resize(frame_rgb: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    """Center-square-crop then resize to size×size. Used as MTCNN fallback."""
    h, w = frame_rgb.shape[:2]
    s = min(h, w)
    top  = (h - s) // 2
    left = (w - s) // 2
    crop = frame_rgb[top:top+s, left:left+s]
    return np.array(Image.fromarray(crop).resize((size, size), Image.BILINEAR))


def extract_faces(video_path: Path):
    """Returns (faces_tensor [N_FRAMES, 3, 224, 224] uint8, stats)."""
    container = av.open(str(video_path))
    all_frames = []
    try:
        for frame in container.decode(video=0):
            all_frames.append(frame.to_ndarray(format="rgb24"))
    finally:
        container.close()

    if len(all_frames) == 0:
        raise RuntimeError(f"No frames decoded from {video_path}")

    indices = _uniform_indices(len(all_frames), N_FRAMES)
    sampled = [all_frames[i] for i in indices]

    out = torch.zeros(N_FRAMES, 3, IMG_SIZE, IMG_SIZE, dtype=torch.uint8)
    detected = 0
    fallback = 0

    if USE_MTCNN:
        # Run MTCNN in a single batched call for efficiency
        pils = [Image.fromarray(f) for f in sampled]
        with torch.no_grad():
            faces = mtcnn(pils)  # returns list of [3,224,224] tensors or None

        for i, face in enumerate(faces):
            if face is not None:
                # MTCNN with post_process=False returns float in [0, 255]
                out[i] = face.clamp(0, 255).byte().cpu()
                detected += 1
            else:
                # fallback: center crop the source frame
                fb = _center_crop_resize(sampled[i])
                out[i] = torch.from_numpy(fb).permute(2, 0, 1).byte()
                fallback += 1
    else:
        for i, frame in enumerate(sampled):
            cc = _center_crop_resize(frame)
            out[i] = torch.from_numpy(cc).permute(2, 0, 1).byte()
            detected += 1   # treated as 'used' here

    stats = {
        "detected": detected,
        "fallback_center_crop": fallback,
        "total_frames": N_FRAMES,
        "source_frame_count": len(all_frames),
    }
    return out, stats


def extract_audio(video_path: Path):
    """Returns audio_tensor [AUDIO_SAMPLES] float32. Returns zeros if no audio stream."""
    try:
        # librosa.load with sr=AUDIO_SR resamples automatically; mono=True averages channels
        wav, _ = librosa.load(str(video_path), sr=AUDIO_SR, mono=True)
    except Exception as e:
        # No audio stream or decode failure
        return torch.zeros(AUDIO_SAMPLES, dtype=torch.float32), False

    wav = wav.astype(np.float32)
    n = wav.shape[0]

    if n >= AUDIO_SAMPLES:
        # center-crop
        start = (n - AUDIO_SAMPLES) // 2
        wav = wav[start:start + AUDIO_SAMPLES]
    else:
        # zero-pad on the right
        pad = np.zeros(AUDIO_SAMPLES - n, dtype=np.float32)
        wav = np.concatenate([wav, pad])

    return torch.from_numpy(wav), True

print("extract_faces() and extract_audio() defined.")

## Phase D — Smoke test on 15 clips

Process 5 clips from each split. Assert that:
- Faces tensor is exactly `[16, 3, 224, 224]` uint8
- Audio tensor is exactly `[160000]` float32
- MTCNN detection rate is reasonable (>50% of frames in most clips; logged if lower)

If anything fails here, the full run will fail too. Fix before launching Phase E.

In [ ]:
# --- Smoke test on 5 per split ---
smoke_samples = []
for s in ("train", "val", "test"):
    sub = manifest[manifest["split"] == s].sample(5, random_state=SEED)
    smoke_samples.append(sub)
smoke_df = pd.concat(smoke_samples, ignore_index=True)

print(f"Smoke-testing on {len(smoke_df)} clips...")

results = []
for _, row in tqdm(smoke_df.iterrows(), total=len(smoke_df)):
    fp = LAVDF_ROOT / row["file"]
    try:
        faces, fstats = extract_faces(fp)
        audio, has_audio = extract_audio(fp)

        # Assertions
        assert faces.shape == (N_FRAMES, 3, IMG_SIZE, IMG_SIZE), f"faces shape {faces.shape}"
        assert faces.dtype == torch.uint8, f"faces dtype {faces.dtype}"
        assert audio.shape == (AUDIO_SAMPLES,), f"audio shape {audio.shape}"
        assert audio.dtype == torch.float32, f"audio dtype {audio.dtype}"

        results.append({
            "file": row["file"], "split": row["split"], "modify_type": row["modify_type"],
            "detected": fstats["detected"], "fallback": fstats["fallback_center_crop"],
            "has_audio": has_audio,
            "audio_rms": float(audio.pow(2).mean().sqrt().item()),
            "ok": True,
        })
    except Exception as e:
        results.append({
            "file": row["file"], "split": row["split"], "modify_type": row["modify_type"],
            "ok": False, "error": str(e)[:80],
        })

res_df = pd.DataFrame(results)
print()
print(res_df.to_string(index=False))

# Summary
ok = res_df["ok"].sum()
n  = len(res_df)
print(f"\nSuccess: {ok}/{n}")
if ok < n:
    print("!! Some clips failed. Investigate before running Phase E.")
else:
    print("\nAll 15 smoke-test clips passed shape assertions. Ready for full run.")
    if "detected" in res_df.columns:
        avg_det = res_df["detected"].mean()
        print(f"Avg MTCNN detection rate: {avg_det:.1f}/{N_FRAMES} frames per clip")

## Phase E — Full preprocessing run

Process all 30,000 clips. **Resumable:** clips with an existing output file are skipped, so re-running this cell after an interruption just continues.

Failures (corrupted clips, unrecoverable decode errors) are logged to `data/preprocessed/_failures.txt` with the error message. Past sessions saw ~7 corrupted files per 25K in LAV-DF — that rate is normal.

**Don't touch the kernel during this.** Save and close any other tabs that might fight for GPU memory.

In [ ]:
# --- Full run ---
FAILURES_FP = PREPROC_ROOT / "_failures.txt"

# Find clips that still need processing
todo = []
for _, row in manifest.iterrows():
    out_fp = PREPROC_ROOT / row["split"] / f"{Path(row['file']).stem}.pt"
    if not out_fp.exists():
        todo.append((row, out_fp))

print(f"Total clips    : {len(manifest):,}")
print(f"Already done   : {len(manifest) - len(todo):,}")
print(f"To process     : {len(todo):,}")

if not todo:
    print("\nNothing to do. All clips already preprocessed.")
else:
    failures = []
    t0 = time.time()
    pbar = tqdm(todo, desc="Preprocessing")

    for row, out_fp in pbar:
        fp = LAVDF_ROOT / row["file"]
        try:
            faces, fstats = extract_faces(fp)
            audio, has_audio = extract_audio(fp)

            entry = {
                "faces":        faces,                       # uint8 [16,3,224,224]
                "audio":        audio,                       # float32 [160000]
                "label_4class": int(row["label_4class"]),
                "label_binary": int(row["label_binary"]),
                "modify_type":  row["modify_type"],
                "split":        row["split"],
                "file":         row["file"],
                "fake_periods": json.loads(row["fake_periods"]),
                "duration":     float(row["duration"]) if pd.notna(row["duration"]) else None,
                "face_stats":   fstats,
                "has_audio":    has_audio,
            }
            torch.save(entry, out_fp)
        except Exception as e:
            failures.append((row["file"], str(e)[:200]))
            pbar.set_postfix(failed=len(failures))

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed/60:.1f} min  ({len(todo)/max(elapsed,1):.1f} clips/sec)")
    print(f"Failures: {len(failures)}")
    if failures:
        with open(FAILURES_FP, "a", encoding="utf-8") as f:
            for fname, err in failures:
                f.write(f"{fname}\t{err}\n")
        print(f"Logged to {FAILURES_FP}")
        print("First few failures:")
        for fname, err in failures[:5]:
            print(f"  {fname}: {err[:120]}")

## Phase F — Verification

Three sanity checks:
1. Count output files per split, compare to manifest counts.
2. Load 10 random `.pt` files and assert every entry's shapes + dtypes.
3. Compute aggregate MTCNN detection stats — flag if many clips had high fallback rates (could indicate bad face detection that hurts training).

In [ ]:
# --- 1. File count check ---
print("--- 1. Output file count per split ---")
for s in ("train", "val", "test"):
    n_manifest = (manifest["split"] == s).sum()
    n_disk     = len(list((PREPROC_ROOT / s).glob("*.pt")))
    flag = "OK" if n_disk == n_manifest else "!! mismatch (check _failures.txt)"
    print(f"  {s:<6}: manifest={n_manifest:>6,}  disk={n_disk:>6,}  [{flag}]")

# --- 2. Random load spot-check ---
print("\n--- 2. Random spot-check (load 10 outputs) ---")
random_files = []
for s in ("train", "val", "test"):
    pool = list((PREPROC_ROOT / s).glob("*.pt"))
    random_files.extend(random.Random(123).sample(pool, min(4, len(pool))))

for fp in random_files[:10]:
    e = torch.load(fp, weights_only=False)
    assert e["faces"].shape == (N_FRAMES, 3, IMG_SIZE, IMG_SIZE)
    assert e["faces"].dtype == torch.uint8
    assert e["audio"].shape == (AUDIO_SAMPLES,)
    assert e["audio"].dtype == torch.float32
    print(f"  OK: {fp.name}  label={e['label_4class']}  "
          f"detected={e['face_stats']['detected']}/{N_FRAMES}  "
          f"audio_rms={e['audio'].pow(2).mean().sqrt():.4f}")

# --- 3. Aggregate face-detection stats ---
print("\n--- 3. MTCNN detection stats across all outputs ---")
det_counts = []
sample_for_stats = []
for s in ("train", "val", "test"):
    pool = list((PREPROC_ROOT / s).glob("*.pt"))
    # Subsample 1000 for speed
    sample_for_stats.extend(random.Random(7).sample(pool, min(1000, len(pool))))

for fp in tqdm(sample_for_stats, desc="Scanning stats"):
    e = torch.load(fp, weights_only=False)
    det_counts.append(e["face_stats"]["detected"])

det = np.array(det_counts)
print(f"  Sample size            : {len(det):,}")
print(f"  Avg frames w/ face     : {det.mean():.2f} / {N_FRAMES}")
print(f"  Median                 : {np.median(det):.0f}")
print(f"  Clips with ≥12/16 faces: {(det >= 12).sum() / len(det):.1%}")
print(f"  Clips with <8/16 faces : {(det < 8).sum() / len(det):.1%}")
print()
print("If <8/16 detection rate is >5%, MTCNN may be struggling with this dataset.")
print("Common causes: profile views, occlusions, dark lighting. Document in paper if notable.")

# --- 4. Disk usage ---
print("\n--- 4. Disk usage ---")
total_bytes = 0
for s in ("train", "val", "test"):
    sz = sum(p.stat().st_size for p in (PREPROC_ROOT / s).glob("*.pt"))
    total_bytes += sz
    print(f"  {s:<6}: {sz/1e9:>6.2f} GB")
print(f"  TOTAL : {total_bytes/1e9:>6.2f} GB")

## Done.

**What's on disk now:**
```
data/preprocessed/
├── train/  (17,400 .pt files)
├── val/    ( 6,900 .pt files)
├── test/   ( 5,700 .pt files)
└── _failures.txt  (if any clips failed)
```

**Update `project_log.md`** with:
- MTCNN detection stats from Phase F (avg frames with face, fallback rate)
- Number of clip failures (if any) and their causes
- Total disk used

**Git commit:**
```cmd
git add notebooks/02_preprocessing.ipynb
git commit -m "nb02: MTCNN face crops + 16kHz audio extraction for LAV-DF 30K subset"
git push origin main
```

**Do NOT commit `data/preprocessed/` — too large.** Make sure it's covered by `.gitignore` (the existing `data/` rule already handles this).

**Next: notebook 03 — model definition (ViViT + WavLM + hierarchical fusion + classification heads).**